In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/achiver7/project-phase-1/data/sc_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/acts_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_val.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/test.json
/kaggle/input/datasets/achiver7/project-phase-1/data/sc_train.jsonl
/kaggle/input/datasets/achiver7/project-phase-1/data/tribunals_train.jsonl
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/sc_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_train.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/acts_val.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tribunals_val_f.bin
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tokenizer.json
/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens/tribunals_train_f.bin
/kaggle/input/datasets/achive

In [2]:
# ============================================================
# FULL ENCODER-DECODER TRANSFORMER + CHECKS
# ============================================================

import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. MULTI-HEAD ATTENTION
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value, mask=None):
        B, Tq, _ = query.shape
        _, Tk, _ = key_value.shape

        Q = self.q_proj(query)
        K = self.k_proj(key_value)
        V = self.v_proj(key_value)

        Q = Q.view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        K = K.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        V = V.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.d_head)

        # mask is additive: 0 for allowed positions, -inf for blocked positions
        if mask is not None:
            scores = scores + mask

        attention = F.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        context = torch.matmul(attention, V)

        context = context.transpose(1, 2).contiguous()
        context = context.view(B, Tq, self.d_model)

        return self.out_proj(context)


# ============================================================
# 2. CAUSAL MASK
# ============================================================

def make_causal_mask(max_len):
    mask = torch.triu(
        torch.ones(max_len, max_len),
        diagonal=1
    )

    mask = mask.masked_fill(mask == 1, float("-inf"))
    mask = mask.masked_fill(mask == 0, 0.0)

    return mask


# ============================================================
# 3. SINUSOIDAL POSITIONAL ENCODING
# ============================================================

def make_sinusoidal_table(max_len, d_model):
    position = torch.arange(
        max_len,
        dtype=torch.float32
    ).unsqueeze(1)

    k = torch.arange(
        0,
        d_model,
        2,
        dtype=torch.float32
    )

    omega = 1.0 / (10000 ** (k / d_model))

    table = torch.zeros(max_len, d_model)

    table[:, 0::2] = torch.sin(position * omega)
    table[:, 1::2] = torch.cos(position * omega)

    return table


# ============================================================
# 4. FEED-FORWARD NETWORK
# ============================================================

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.dropout(x)
        return x


# ============================================================
# 5. ENCODER LAYER
# POST-LN
# ============================================================

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # Self-attention -> residual -> LayerNorm
        x = self.norm1(
            x + self.dropout(
                self.self_attn(x, x)
            )
        )

        # FFN -> residual -> LayerNorm
        x = self.norm2(
            x + self.ffn(x)
        )

        return x


# ============================================================
# 6. ENCODER
# ============================================================

class TransformerEncoder(nn.Module):
    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerEncoderLayer(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        return x


# ============================================================
# 7. DECODER LAYER
# THREE POST-LN SUBLAYERS
# ============================================================

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.cross_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, causal_mask):

        # 1. Causal self-attention
        x = self.norm1(
            x + self.dropout(
                self.self_attn(
                    x,
                    x,
                    mask=causal_mask
                )
            )
        )

        # 2. Cross-attention
        # Q = decoder, K/V = encoder memory
        x = self.norm2(
            x + self.dropout(
                self.cross_attn(
                    x,
                    memory
                )
            )
        )

        # 3. FFN
        x = self.norm3(
            x + self.ffn(x)
        )

        return x


# ============================================================
# 8. DECODER
# ============================================================

class TransformerDecoder(nn.Module):
    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        max_len=1024,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerDecoderLayer(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.register_buffer(
            "causal_mask",
            make_causal_mask(max_len),
            persistent=False
        )

    def forward(self, x, memory):

        T = x.size(1)

        mask = self.causal_mask[:T, :T]

        for layer in self.layers:
            x = layer(
                x,
                memory,
                mask
            )

        return x


# ============================================================
# 9. COMPLETE ENCODER-DECODER MODEL
# ============================================================

class TransformerModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_src_len,
        max_tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

        # Shared token embedding
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Fixed sinusoidal positional encodings
        self.register_buffer(
            "src_pos_encoding",
            make_sinusoidal_table(
                max_src_len,
                d_model
            ),
            persistent=False
        )

        self.register_buffer(
            "tgt_pos_encoding",
            make_sinusoidal_table(
                max_tgt_len,
                d_model
            ),
            persistent=False
        )

        self.dropout = nn.Dropout(dropout)

        self.encoder = TransformerEncoder(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            dropout=dropout
        )

        self.decoder = TransformerDecoder(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            max_len=max_tgt_len,
            dropout=dropout
        )

        # No bias: exact tied vocabulary projection
        self.output_projection = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Tie output projection to token embedding
        self.output_projection.weight = self.token_embedding.weight

        self._init_weights()

    def _init_weights(self):

        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02
        )

        for module in self.modules():

            if isinstance(module, nn.Linear):

                if module.weight is not self.token_embedding.weight:
                    nn.init.xavier_uniform_(module.weight)

                if module.bias is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.LayerNorm):

                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def encode(self, src):

        B, T_src = src.shape

        assert T_src <= self.max_src_len

        # Original Transformer-style sqrt(d) scaling
        x = self.token_embedding(src)
        x = x * math.sqrt(self.d_model)

        # Add fixed positional encoding
        x = x + self.src_pos_encoding[:T_src].unsqueeze(0)

        x = self.dropout(x)

        return self.encoder(x)

    def decode(self, tgt_in, memory):

        B, T_tgt = tgt_in.shape

        assert T_tgt <= self.max_tgt_len

        # Token embedding + sqrt(d) scaling
        x = self.token_embedding(tgt_in)
        x = x * math.sqrt(self.d_model)

        # Add fixed positional encoding
        x = x + self.tgt_pos_encoding[:T_tgt].unsqueeze(0)

        x = self.dropout(x)

        x = self.decoder(
            x,
            memory
        )

        # Raw logits; no softmax
        return self.output_projection(x)

    def forward(self, src, tgt_in):

        memory = self.encode(src)

        return self.decode(
            tgt_in,
            memory
        )


# ============================================================
# 10. BUILD_MODEL
# ============================================================

def build_model(vocab_size, src_len, tgt_len):

    assert src_len + tgt_len == 1024

    return TransformerModel(
        vocab_size=vocab_size,
        max_src_len=src_len,
        max_tgt_len=tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    )


# ============================================================
# 11. CHECKS
# ============================================================

dev = "cuda" if torch.cuda.is_available() else "cpu"

results = []


def check(name, fn):

    try:
        msg = fn()

        print(
            f"PASS  {name}" +
            (f"  | {msg}" if msg else "")
        )

        results.append(True)

    except AssertionError as e:

        print(
            f"FAIL  {name}  | {e}"
        )

        results.append(False)

    except Exception as e:

        print(
            f"ERROR {name}  | "
            f"{type(e).__name__}: {e}"
        )

        results.append(False)


# ============================================================
# 12. SINUSOIDAL TABLE TEST
# ============================================================

def t_sin():

    L, d = 50, 16

    t = make_sinusoidal_table(L, d)

    assert t.shape == (L, d), \
        f"shape {tuple(t.shape)}"

    assert torch.allclose(
        t[0, 0::2],
        torch.zeros(d // 2)
    ), "pos 0 sin dims should be 0"

    assert torch.allclose(
        t[0, 1::2],
        torch.ones(d // 2)
    ), "pos 0 cos dims should be 1"

    i, k = 7, 3

    w = 1.0 / (
        10000 ** (2 * k / d)
    )

    assert abs(
        t[i, 2 * k].item()
        - math.sin(w * i)
    ) < 1e-5, \
        "sin entry differs from slide formula"

    assert abs(
        t[i, 2 * k + 1].item()
        - math.cos(w * i)
    ) < 1e-5, \
        "cos entry differs from slide formula"


check(
    "sinusoidal table matches slide formula",
    t_sin
)


# ============================================================
# 13. ATTENTION SHAPE TEST
# ============================================================

def t_attn_shape():

    m = MultiHeadAttention(
        64,
        4
    ).eval()

    q = torch.randn(
        2,
        5,
        64
    )

    kv = torch.randn(
        2,
        9,
        64
    )

    assert m(q, kv).shape == (
        2,
        5,
        64
    ), "cross-attention (Tq != Tk) output shape"


check(
    "attention output shape (Tq != Tk)",
    t_attn_shape
)


# ============================================================
# 14. ATTENTION MASK TEST
# ============================================================

def t_attn_mask():

    torch.manual_seed(0)

    m = MultiHeadAttention(
        64,
        4
    ).eval()

    T = 8

    x = torch.randn(
        1,
        T,
        64
    )

    mask = make_causal_mask(T)

    y1 = m(
        x,
        x,
        mask=mask
    )

    x2 = x.clone()

    x2[:, 5:] = torch.randn(
        1,
        T - 5,
        64
    )

    y2 = m(
        x2,
        x2,
        mask=mask
    )

    d = (
        y1[:, :5]
        - y2[:, :5]
    ).abs().max().item()

    assert d < 1e-5, (
        f"outputs at positions 0-4 changed by "
        f"{d:.2e} when only positions 5+ changed. "
        f"Mask convention is probably inverted."
    )

    x3 = x.clone()

    x3[:, 0] = torch.randn(
        1,
        64
    )

    y3 = m(
        x3,
        x3,
        mask=mask
    )

    assert (
        y1[:, 7] - y3[:, 7]
    ).abs().max().item() > 1e-6, \
        "last position ignores earlier tokens"


check(
    "attention + causal mask (no future leak, past visible)",
    t_attn_mask
)


# ============================================================
# 15. SMALL MODEL
# ============================================================

SMALL = dict(
    vocab_size=100,
    max_src_len=32,
    max_tgt_len=32,
    d_model=64,
    n_heads=4,
    d_ff=256,
    n_layers=2,
    dropout=0.1
)

V = SMALL["vocab_size"]

small = TransformerModel(
    **SMALL
).to(dev)


# ============================================================
# 16. MODEL OUTPUT SHAPE
# ============================================================

def t_shape():

    small.eval()

    s = torch.randint(
        0,
        V,
        (3, 20),
        device=dev
    )

    t = torch.randint(
        0,
        V,
        (3, 12),
        device=dev
    )

    with torch.no_grad():

        o = small(
            s,
            t
        )

    assert o.shape == (
        3,
        12,
        V
    ), f"got {tuple(o.shape)}"

    assert torch.isfinite(o).all(), \
        "NaN/inf in logits"


check(
    "model output shape + finite",
    t_shape
)


# ============================================================
# 17. FULL MODEL CAUSALITY
# ============================================================

def t_causal():

    small.eval()

    s = torch.randint(
        0,
        V,
        (2, 20),
        device=dev
    )

    a = torch.randint(
        0,
        V,
        (2, 16),
        device=dev
    )

    b = a.clone()

    b[:, 9:] = torch.randint(
        0,
        V,
        b[:, 9:].shape,
        device=dev
    )

    with torch.no_grad():

        d = (
            small(s, a)[:, :9]
            - small(s, b)[:, :9]
        ).abs().max().item()

    assert d < 1e-5, \
        f"decoder leaks the future (diff {d:.2e})"


check(
    "decoder causality (full model)",
    t_causal
)


# ============================================================
# 18. ENCODER DEPENDENCE
# ============================================================

def t_enc_dep():

    small.eval()

    s = torch.randint(
        0,
        V,
        (2, 20),
        device=dev
    )

    t = torch.randint(
        0,
        V,
        (2, 12),
        device=dev
    )

    s2 = s.clone()

    s2[:, 0] = (
        s2[:, 0] + 1
    ) % V

    with torch.no_grad():

        d = (
            small(s, t)
            - small(s2, t)
        ).abs().max().item()

    assert d > 1e-7, \
        "changing src does not change output (cross-attention not connected?)"

    # Encoder must be bidirectional
    s3 = s.clone()

    s3[:, -1] = (
        s3[:, -1] + 1
    ) % V

    with torch.no_grad():

        m1 = small.encode(s)
        m2 = small.encode(s3)

    assert (
        m1[:, 0] - m2[:, 0]
    ).abs().max().item() > 1e-7, \
        "encoder self-attention looks causal (should be unmasked)"


check(
    "encoder dependence + bidirectional encoder",
    t_enc_dep
)


# ============================================================
# 19. DROPOUT MODES
# ============================================================

def t_modes():

    s = torch.randint(
        0,
        V,
        (2, 20),
        device=dev
    )

    t = torch.randint(
        0,
        V,
        (2, 12),
        device=dev
    )

    small.eval()

    with torch.no_grad():

        assert torch.equal(
            small(s, t),
            small(s, t)
        ), "eval mode not deterministic"

    small.train()

    with torch.no_grad():

        assert not torch.equal(
            small(s, t),
            small(s, t)
        ), "dropout seems inactive in train mode"

    small.eval()


check(
    "dropout off in eval / on in train",
    t_modes
)


# ============================================================
# 20. OUTPUT HEAD TIE
# ============================================================

def t_tie():

    assert (
        small.output_projection.weight.data_ptr()
        ==
        small.token_embedding.weight.data_ptr()
    ), "head not tied"

    n_with = sum(
        p.numel()
        for p in small.parameters()
    )

    assert n_with == (
        len({
            id(p)
            for p in small.parameters()
        }) * 0
        + n_with
    )


check(
    "output head tied to embedding",
    t_tie
)


# ============================================================
# 21. SMALL MODEL PARAMETER COUNT
# ============================================================

def t_params():

    c = SMALL

    d = c["d_model"]
    ff = c["d_ff"]
    L = c["n_layers"]
    v = c["vocab_size"]

    attn = 4 * (
        d * d + d
    )

    ffn = (
        2 * d * ff
        + ff
        + d
    )

    enc = (
        attn
        + ffn
        + 2 * (2 * d)
    )

    dec = (
        2 * attn
        + ffn
        + 3 * (2 * d)
    )

    exp = (
        v * d
        + L * enc
        + L * dec
    )

    got = sum(
        p.numel()
        for p in small.parameters()
    )

    assert got == exp, (
        f"got {got:,}, "
        f"hand formula {exp:,} "
        f"(diff {got-exp:+,}). "
        f"Missing/extra bias, norm, or untied head?"
    )

    return f"small={got:,}"


check(
    "parameter count = hand formula (small config)",
    t_params
)


# ============================================================
# 22. GRADIENT TEST
# ============================================================

def t_grad():

    small.train()

    s = torch.randint(
        0,
        V,
        (2, 20),
        device=dev
    )

    t = torch.randint(
        0,
        V,
        (2, 12),
        device=dev
    )

    y = torch.randint(
        0,
        V,
        (2, 12),
        device=dev
    )

    small.zero_grad()

    logits = small(
        s,
        t
    )

    loss = F.cross_entropy(
        logits.reshape(-1, V),
        y.reshape(-1)
    )

    loss.backward()

    dead = [
        n
        for n, p in small.named_parameters()
        if p.grad is None
        or p.grad.abs().sum() == 0
    ]

    assert not dead, \
        f"no gradient reached: {dead[:5]}"

    small.eval()


check(
    "every parameter receives gradient",
    t_grad
)


# ============================================================
# 23. OVERFIT ONE TINY BATCH
# ============================================================

def t_overfit():

    torch.manual_seed(0)

    m = TransformerModel(
        **{
            **SMALL,
            "dropout": 0.0
        }
    ).to(dev).train()

    opt = torch.optim.AdamW(
        m.parameters(),
        lr=3e-3
    )

    s = torch.randint(
        0,
        V,
        (4, 20),
        device=dev
    )

    t = torch.randint(
        0,
        V,
        (4, 12),
        device=dev
    )

    y = torch.randint(
        0,
        V,
        (4, 12),
        device=dev
    )

    first = None

    for i in range(150):

        loss = F.cross_entropy(
            m(s, t).reshape(-1, V),
            y.reshape(-1)
        )

        opt.zero_grad()
        loss.backward()
        opt.step()

        first = (
            loss.item()
            if first is None
            else first
        )

    assert loss.item() < 0.5 * first, (
        f"loss {first:.2f} -> {loss.item():.2f}; "
        f"cannot memorize one batch (bug in model)"
    )

    return (
        f"loss {first:.2f} -> "
        f"{loss.item():.3f}"
    )


check(
    "overfits one tiny batch",
    t_overfit
)


# ============================================================
# 24. EMBEDDING / POSITION INFO
# ============================================================

def info():

    e = (
        small.token_embedding.weight
        .std()
        .item()
    )

    p = (
        small.src_pos_encoding
        .std()
        .item()
    )

    print(
        f"INFO  token-embedding std {e:.4f} | "
        f"positional-encoding std {p:.4f} | "
        f"ratio pos/emb = {p/e:.1f}x"
    )


info()


# ============================================================
# 25. FINAL SMALL-MODEL RESULT
# ============================================================

print(
    f"\n{sum(results)}/{len(results)} checks passed"
)

print(
    "Next: build REAL config, print parameter count "
    "(expected 129,957,888), then run scaffold preflight."
)


# ============================================================
# 26. REAL MODEL
# ============================================================

# Uses the scaffold's CFG if it already exists.
# SRC_LEN + TGT_LEN must equal 1024.

if "CFG" in globals():

    assert CFG["SRC_LEN"] + CFG["TGT_LEN"] == 1024

    model = build_model(
        CFG["VOCAB_SIZE"],
        CFG["SRC_LEN"],
        CFG["TGT_LEN"]
    ).to(dev)

    n_params = sum(
        p.numel()
        for p in model.parameters()
    )

    print(
        "\nREAL MODEL"
    )

    print(
        f"Parameters: {n_params:,}"
    )

    print(
        f"Parameters: {n_params / 1e6:.2f}M"
    )

    print(
        "Hand calculation: 129,957,888"
    )

    assert n_params == 129_957_888, (
        f"Expected 129,957,888 parameters, "
        f"got {n_params:,}"
    )

    assert (
        50_000_000
        <= n_params
        <= 300_000_000
    ), (
        f"Parameter count {n_params:,} "
        f"is outside the 50M-300M rule."
    )

    print(
        "PASS  parameter count = 129,957,888"
    )

    print(
        "PASS  model is inside 50M-300M range"
    )

PASS  sinusoidal table matches slide formula
PASS  attention output shape (Tq != Tk)
PASS  attention + causal mask (no future leak, past visible)
PASS  model output shape + finite
PASS  decoder causality (full model)
PASS  encoder dependence + bidirectional encoder
PASS  dropout off in eval / on in train
PASS  output head tied to embedding
PASS  parameter count = hand formula (small config)  | small=239,872
PASS  every parameter receives gradient
PASS  overfits one tiny batch  | loss 4.61 -> 0.007
INFO  token-embedding std 0.0202 | positional-encoding std 0.5758 | ratio pos/emb = 28.6x

11/11 checks passed
Next: build REAL config, print parameter count (expected 129,957,888), then run scaffold preflight.


In [3]:
# ============================================================
# CHECKS for your encoder-decoder (tests only; contains no model code).
# Run in a cell AFTER your classes are defined:
#   MultiHeadAttention, make_causal_mask, make_sinusoidal_table, TransformerModel
# Each check prints PASS/FAIL with a hint and keeps going.
# ============================================================
import math, torch, torch.nn.functional as F

dev = "cuda" if torch.cuda.is_available() else "cpu"
results = []


def check(name, fn):
    try:
        msg = fn()
        print(f"PASS  {name}" + (f"  | {msg}" if msg else ""))
        results.append(True)
    except AssertionError as e:
        print(f"FAIL  {name}  | {e}")
        results.append(False)
    except Exception as e:
        print(f"ERROR {name}  | {type(e).__name__}: {e}")
        results.append(False)


# ---------- 1. sinusoidal table vs slide formula ----------
def t_sin():
    L, d = 50, 16
    t = make_sinusoidal_table(L, d)
    assert t.shape == (L, d), f"shape {tuple(t.shape)}"
    assert torch.allclose(t[0, 0::2], torch.zeros(d // 2)), "pos 0 sin dims should be 0"
    assert torch.allclose(t[0, 1::2], torch.ones(d // 2)), "pos 0 cos dims should be 1"
    i, k = 7, 3  # token position, frequency index
    w = 1.0 / (10000 ** (2 * k / d))
    assert abs(t[i, 2 * k].item() - math.sin(w * i)) < 1e-5, "sin entry differs from slide formula"
    assert abs(t[i, 2 * k + 1].item() - math.cos(w * i)) < 1e-5, "cos entry differs from slide formula"
check("sinusoidal table matches slide formula", t_sin)


# ---------- 2. attention: shapes, mask convention ----------
def t_attn_shape():
    m = MultiHeadAttention(64, 4).eval()
    q, kv = torch.randn(2, 5, 64), torch.randn(2, 9, 64)
    assert m(q, kv).shape == (2, 5, 64), "cross-attention (Tq != Tk) output shape"
check("attention output shape (Tq != Tk)", t_attn_shape)


def t_attn_mask():
    torch.manual_seed(0)
    m = MultiHeadAttention(64, 4).eval()
    T = 8
    x = torch.randn(1, T, 64)
    mask = make_causal_mask(T)
    y1 = m(x, x, mask=mask)
    x2 = x.clone()
    x2[:, 5:] = torch.randn(1, T - 5, 64)          # change only FUTURE tokens
    y2 = m(x2, x2, mask=mask)
    d = (y1[:, :5] - y2[:, :5]).abs().max().item()
    assert d < 1e-5, (f"outputs at positions 0-4 changed by {d:.2e} when only positions 5+ changed. "
                      "Mask convention is probably inverted: make_causal_mask is ADDITIVE (0 / -inf) "
                      "but attention may treat it as a keep-mask (mask == 0).")
    # and positions must still see the past: changing an early token must change later outputs
    x3 = x.clone(); x3[:, 0] = torch.randn(1, 64)
    y3 = m(x3, x3, mask=mask)
    assert (y1[:, 7] - y3[:, 7]).abs().max().item() > 1e-6, "last position ignores earlier tokens"
check("attention + causal mask (no future leak, past visible)", t_attn_mask)


# ---------- 3. full small model ----------
SMALL = dict(vocab_size=100, max_src_len=32, max_tgt_len=32, d_model=64,
             n_heads=4, d_ff=256, n_layers=2, dropout=0.1)
V = SMALL["vocab_size"]
small = TransformerModel(**SMALL).to(dev)


def t_shape():
    small.eval()
    s = torch.randint(0, V, (3, 20), device=dev); t = torch.randint(0, V, (3, 12), device=dev)
    with torch.no_grad():
        o = small(s, t)
    assert o.shape == (3, 12, V), f"got {tuple(o.shape)}"
    assert torch.isfinite(o).all(), "NaN/inf in logits"
check("model output shape + finite", t_shape)


def t_causal():
    small.eval()
    s = torch.randint(0, V, (2, 20), device=dev); a = torch.randint(0, V, (2, 16), device=dev)
    b = a.clone(); b[:, 9:] = torch.randint(0, V, b[:, 9:].shape, device=dev)
    with torch.no_grad():
        d = (small(s, a)[:, :9] - small(s, b)[:, :9]).abs().max().item()
    assert d < 1e-5, f"decoder leaks the future (diff {d:.2e})"
check("decoder causality (full model)", t_causal)


def t_enc_dep():
    small.eval()
    s = torch.randint(0, V, (2, 20), device=dev); t = torch.randint(0, V, (2, 12), device=dev)
    s2 = s.clone(); s2[:, 0] = (s2[:, 0] + 1) % V
    with torch.no_grad():
        d = (small(s, t) - small(s2, t)).abs().max().item()
    assert d > 1e-7, "changing src does not change output (cross-attention not connected?)"
    # encoder must be bidirectional: changing the LAST src token changes memory at position 0
    s3 = s.clone(); s3[:, -1] = (s3[:, -1] + 1) % V
    with torch.no_grad():
        m1, m2 = small.encode(s), small.encode(s3)
    assert (m1[:, 0] - m2[:, 0]).abs().max().item() > 1e-7, "encoder self-attention looks causal (should be unmasked)"
check("encoder dependence + bidirectional encoder", t_enc_dep)


def t_modes():
    s = torch.randint(0, V, (2, 20), device=dev); t = torch.randint(0, V, (2, 12), device=dev)
    small.eval()
    with torch.no_grad():
        assert torch.equal(small(s, t), small(s, t)), "eval mode not deterministic"
    small.train()
    with torch.no_grad():
        assert not torch.equal(small(s, t), small(s, t)), "dropout seems inactive in train mode"
    small.eval()
check("dropout off in eval / on in train", t_modes)


def t_tie():
    assert small.output_projection.weight.data_ptr() == small.token_embedding.weight.data_ptr(), "head not tied"
    n_with = sum(p.numel() for p in small.parameters())
    assert n_with == len({id(p) for p in small.parameters()}) * 0 + n_with  # parameters() already de-duplicates
check("output head tied to embedding", t_tie)


def t_params():
    c = SMALL; d, ff, L, v = c["d_model"], c["d_ff"], c["n_layers"], c["vocab_size"]
    attn = 4 * (d * d + d)
    ffn = 2 * d * ff + ff + d
    enc = attn + ffn + 2 * (2 * d)
    dec = 2 * attn + ffn + 3 * (2 * d)
    exp = v * d + L * enc + L * dec
    got = sum(p.numel() for p in small.parameters())
    assert got == exp, f"got {got:,}, hand formula {exp:,} (diff {got-exp:+,}). Missing/extra bias, norm, or untied head?"
    return f"small={got:,}"
check("parameter count = hand formula (small config)", t_params)


def t_grad():
    small.train()
    s = torch.randint(0, V, (2, 20), device=dev); t = torch.randint(0, V, (2, 12), device=dev)
    y = torch.randint(0, V, (2, 12), device=dev)
    small.zero_grad()
    F.cross_entropy(small(s, t).reshape(-1, V), y.reshape(-1)).backward()
    dead = [n for n, p in small.named_parameters() if p.grad is None or p.grad.abs().sum() == 0]
    assert not dead, f"no gradient reached: {dead[:5]}"
    small.eval()
check("every parameter receives gradient", t_grad)


def t_overfit():
    torch.manual_seed(0)
    m = TransformerModel(**{**SMALL, "dropout": 0.0}).to(dev).train()
    opt = torch.optim.AdamW(m.parameters(), lr=3e-3)
    s = torch.randint(0, V, (4, 20), device=dev); t = torch.randint(0, V, (4, 12), device=dev)
    y = torch.randint(0, V, (4, 12), device=dev)
    first = None
    for i in range(150):
        loss = F.cross_entropy(m(s, t).reshape(-1, V), y.reshape(-1))
        opt.zero_grad(); loss.backward(); opt.step()
        first = first if first is not None else loss.item()
    assert loss.item() < 0.5 * first, f"loss {first:.2f} -> {loss.item():.2f}; cannot memorize one batch (bug in model)"
    return f"loss {first:.2f} -> {loss.item():.3f}"
check("overfits one tiny batch", t_overfit)


# ---------- info (not pass/fail): size of embedding vs position signal ----------
def info():
    e = small.token_embedding.weight.std().item()
    p = small.src_pos_encoding.std().item()
    print(f"INFO  token-embedding std {e:.4f} | positional-encoding std {p:.4f} | ratio pos/emb = {p/e:.1f}x"
          "  (your decision: scale embeddings or not; record it)")
info()

print(f"\n{sum(results)}/{len(results)} checks passed")
print("Next: build your REAL config, print the parameter count (hand formula, V=40000, 6+6 layers: 129,957,888),")
print("then run the scaffold's preflight.")

PASS  sinusoidal table matches slide formula
PASS  attention output shape (Tq != Tk)
PASS  attention + causal mask (no future leak, past visible)
PASS  model output shape + finite
PASS  decoder causality (full model)
PASS  encoder dependence + bidirectional encoder
PASS  dropout off in eval / on in train
PASS  output head tied to embedding
PASS  parameter count = hand formula (small config)  | small=239,872
PASS  every parameter receives gradient
PASS  overfits one tiny batch  | loss 4.61 -> 0.007
INFO  token-embedding std 0.0202 | positional-encoding std 0.5758 | ratio pos/emb = 28.6x  (your decision: scale embeddings or not; record it)

11/11 checks passed
Next: build your REAL config, print the parameter count (hand formula, V=40000, 6+6 layers: 129,957,888),
then run the scaffold's preflight.


In [4]:
CFG = {}

CFG["VOCAB_SIZE"] = 40000
CFG["SRC_LEN"] = 768
CFG["TGT_LEN"] = 256

assert CFG["SRC_LEN"] + CFG["TGT_LEN"] == 1024

print(CFG)

{'VOCAB_SIZE': 40000, 'SRC_LEN': 768, 'TGT_LEN': 256}


In [5]:
model = build_model(
    CFG["VOCAB_SIZE"],
    CFG["SRC_LEN"],
    CFG["TGT_LEN"]
).to(dev)

n_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Parameters: {n_params:,}")
print(f"Parameters: {n_params / 1e6:.2f}M")

assert n_params == 129_957_888
assert 50_000_000 <= n_params <= 300_000_000

print("PASS  parameter count = 129,957,888")
print("PASS  model is inside 50M-300M range")

Parameters: 129,957,888
Parameters: 129.96M
PASS  parameter count = 129,957,888
PASS  model is inside 50M-300M range


In [6]:
# ============================================================
# REAL MODEL PREFLIGHT
# ============================================================

model.eval()

B = 2
T_src = CFG["SRC_LEN"]
T_tgt = CFG["TGT_LEN"]
V = CFG["VOCAB_SIZE"]

src = torch.randint(
    0, V,
    (B, T_src),
    device=dev
)

tgt_in = torch.randint(
    0, V,
    (B, T_tgt),
    device=dev
)

targets = torch.randint(
    0, V,
    (B, T_tgt),
    device=dev
)

# ------------------------------------------------------------
# 1. Output shape
# ------------------------------------------------------------

with torch.no_grad():
    logits = model(src, tgt_in)

expected = (B, T_tgt, V)

print("Output shape:", tuple(logits.shape))

assert logits.shape == expected, (
    f"Expected {expected}, got {tuple(logits.shape)}"
)

assert torch.isfinite(logits).all(), "NaN/inf in logits"

print("PASS  output shape + finite")


# ------------------------------------------------------------
# 2. Decoder causality
# ------------------------------------------------------------

tgt_a = tgt_in.clone()
tgt_b = tgt_in.clone()

check_pos = T_tgt // 2

tgt_b[:, check_pos + 1:] = torch.randint(
    0,
    V,
    tgt_b[:, check_pos + 1:].shape,
    device=dev
)

with torch.no_grad():
    logits_a = model(src, tgt_a)
    logits_b = model(src, tgt_b)

diff = (
    logits_a[:, :check_pos + 1]
    - logits_b[:, :check_pos + 1]
).abs().max().item()

print(
    f"Decoder causality max diff: {diff:.3e}"
)

assert diff < 1e-5, (
    "Decoder causality failed. "
    "Check the causal mask convention."
)

print("PASS  decoder causality")


# ------------------------------------------------------------
# 3. Encoder dependence
# ------------------------------------------------------------

src_changed = src.clone()

src_changed[:, 0] = (
    src_changed[:, 0] + 1
) % V

with torch.no_grad():
    logits_original = model(src, tgt_in)
    logits_changed = model(src_changed, tgt_in)

diff = (
    logits_original - logits_changed
).abs().max().item()

print(
    f"Encoder dependence max diff: {diff:.3e}"
)

assert diff > 1e-7, (
    "Changing src did not change output. "
    "Check encoder/cross-attention."
)

print("PASS  encoder dependence")


# ------------------------------------------------------------
# 4. Initial loss
# ------------------------------------------------------------

with torch.no_grad():
    logits = model(src, tgt_in)

loss = F.cross_entropy(
    logits.reshape(-1, V),
    targets.reshape(-1)
)

print(
    f"Initial loss: {loss.item():.4f}"
)

# ln(40000) ≈ 10.5966
assert 9.0 < loss.item() < 12.0, (
    f"Initial loss {loss.item():.4f} is not near 10.6. "
    "Check initialization, logits, vocabulary size, "
    "or target/loss construction."
)

print("PASS  initial loss near 10.6")


Output shape: (2, 256, 40000)
PASS  output shape + finite
Decoder causality max diff: 0.000e+00
PASS  decoder causality
Encoder dependence max diff: 8.428e-03
PASS  encoder dependence
Initial loss: 10.7755
PASS  initial loss near 10.6


In [7]:
# ============================================================
# REAL MODEL BENCHMARK
# ============================================================

import time
import torch

model.train()

B = 2
T_src = CFG["SRC_LEN"]
T_tgt = CFG["TGT_LEN"]
V = CFG["VOCAB_SIZE"]

src = torch.randint(
    0, V,
    (B, T_src),
    device=dev
)

tgt_in = torch.randint(
    0, V,
    (B, T_tgt),
    device=dev
)

# ------------------------------------------------------------
# GPU memory before benchmark
# ------------------------------------------------------------

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# ------------------------------------------------------------
# Warmup
# ------------------------------------------------------------

for _ in range(5):
    with torch.no_grad():
        _ = model(src, tgt_in)

if torch.cuda.is_available():
    torch.cuda.synchronize()

# ------------------------------------------------------------
# Timed benchmark
# ------------------------------------------------------------

N = 20

if torch.cuda.is_available():
    torch.cuda.synchronize()

start = time.perf_counter()

for _ in range(N):
    with torch.no_grad():
        _ = model(src, tgt_in)

if torch.cuda.is_available():
    torch.cuda.synchronize()

elapsed = time.perf_counter() - start

# Count source + target tokens processed
tokens = N * B * (T_src + T_tgt)

tokens_per_sec = tokens / elapsed

print(f"Benchmark iterations : {N}")
print(f"Elapsed time         : {elapsed:.3f} s")
print(f"Tokens processed     : {tokens:,}")
print(f"Tokens/sec           : {tokens_per_sec:,.0f}")

# ------------------------------------------------------------
# Peak GPU memory
# ------------------------------------------------------------

if torch.cuda.is_available():

    peak_allocated = (
        torch.cuda.max_memory_allocated()
        / (1024 ** 3)
    )

    peak_reserved = (
        torch.cuda.max_memory_reserved()
        / (1024 ** 3)
    )

    print(
        f"Peak GPU allocated   : "
        f"{peak_allocated:.2f} GB"
    )

    print(
        f"Peak GPU reserved    : "
        f"{peak_reserved:.2f} GB"
    )

else:
    print("GPU not available; memory benchmark skipped.")

Benchmark iterations : 20
Elapsed time         : 1.624 s
Tokens processed     : 40,960
Tokens/sec           : 25,228
Peak GPU allocated   : 1.02 GB
Peak GPU reserved    : 1.16 GB


In [8]:
import time, torch, torch.nn.functional as F

model.train()
V, S, T = CFG["VOCAB_SIZE"], CFG["SRC_LEN"], CFG["TGT_LEN"]
MB = 1                                   # use your CFG micro-batch
opt = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=0.1)
scaler = torch.cuda.amp.GradScaler()

def step():
    src = torch.randint(0, V, (MB, S), device=dev)
    tgt_in = torch.randint(0, V, (MB, T), device=dev)
    y = torch.randint(0, V, (MB, T), device=dev)
    with torch.autocast("cuda", dtype=torch.float16):
        loss = F.cross_entropy(model(src, tgt_in).float().reshape(-1, V), y.reshape(-1))
    opt.zero_grad(set_to_none=True)
    scaler.scale(loss).backward()
    scaler.unscale_(opt)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(opt); scaler.update()

for _ in range(5): step()                 # warmup
torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats()
N = 30; t0 = time.perf_counter()
for _ in range(N): step()
torch.cuda.synchronize(); dt = time.perf_counter() - t0
print(f"{N*MB*(S+T)/dt:,.0f} window-tokens/s | peak alloc {torch.cuda.max_memory_allocated()/2**30:.2f} GB")

/tmp/ipykernel_23/2145039163.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler()


9,578 window-tokens/s | peak alloc 3.25 GB


In [9]:
# ============================================================
# PHASE 1 - ARCHITECTURE 2 (encoder-decoder) TRAINING SCAFFOLD
#
# Scaffolding only: data windows, split into encoder input / decoder target,
# loss, validation, checkpointing, time guard, preflight tests and a
# generation helper. The model section (section 0) is the author's own code.
#
# Model contract (the only thing this file needs from your model):
#     model(src, tgt_in) -> logits
#         src     : LongTensor [B, SRC_LEN]   encoder input (the "prompt" part)
#         tgt_in  : LongTensor [B, TGT_LEN]   decoder input (<bos> + target shifted right)
#         logits  : FloatTensor [B, TGT_LEN, V]
#     Decoder self-attention must be causal. Encoder self-attention may see
#     all of src. Cross-attention connects decoder queries to encoder outputs.
#
# CFG optimizer/schedule values are copied from Architecture 1 so the two runs
# are comparable. They are yours to change; SRC_LEN / TGT_LEN are left for you.
# ============================================================
import os, time, math, json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tokenizers import Tokenizer

# ============================================================
# 0. YOUR MODEL
# ============================================================
# ---- the author's own model code (unchanged) ----
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


# ============================================================
# 1. MULTI-HEAD ATTENTION
# ============================================================

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, n_heads, dropout=0.0):
        super().__init__()

        assert d_model % n_heads == 0

        self.d_model = d_model
        self.n_heads = n_heads
        self.d_head = d_model // n_heads

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, query, key_value, mask=None):
        B, Tq, _ = query.shape
        _, Tk, _ = key_value.shape

        Q = self.q_proj(query)
        K = self.k_proj(key_value)
        V = self.v_proj(key_value)

        Q = Q.view(B, Tq, self.n_heads, self.d_head).transpose(1, 2)
        K = K.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)
        V = V.view(B, Tk, self.n_heads, self.d_head).transpose(1, 2)

        scores = torch.matmul(Q, K.transpose(-2, -1))
        scores = scores / math.sqrt(self.d_head)

        # mask is additive: 0 for allowed positions, -inf for blocked positions
        if mask is not None:
            scores = scores + mask

        attention = F.softmax(scores, dim=-1)
        attention = self.dropout(attention)

        context = torch.matmul(attention, V)

        context = context.transpose(1, 2).contiguous()
        context = context.view(B, Tq, self.d_model)

        return self.out_proj(context)


# ============================================================
# 2. CAUSAL MASK
# ============================================================

def make_causal_mask(max_len):
    mask = torch.triu(
        torch.ones(max_len, max_len),
        diagonal=1
    )

    mask = mask.masked_fill(mask == 1, float("-inf"))
    mask = mask.masked_fill(mask == 0, 0.0)

    return mask


# ============================================================
# 3. SINUSOIDAL POSITIONAL ENCODING
# ============================================================

def make_sinusoidal_table(max_len, d_model):
    position = torch.arange(
        max_len,
        dtype=torch.float32
    ).unsqueeze(1)

    k = torch.arange(
        0,
        d_model,
        2,
        dtype=torch.float32
    )

    omega = 1.0 / (10000 ** (k / d_model))

    table = torch.zeros(max_len, d_model)

    table[:, 0::2] = torch.sin(position * omega)
    table[:, 1::2] = torch.cos(position * omega)

    return table


# ============================================================
# 4. FEED-FORWARD NETWORK
# ============================================================

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout=0.1):
        super().__init__()

        self.linear1 = nn.Linear(d_model, d_ff)
        self.activation = nn.GELU()
        self.linear2 = nn.Linear(d_ff, d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        x = self.linear1(x)
        x = self.activation(x)
        x = self.linear2(x)
        x = self.dropout(x)
        return x


# ============================================================
# 5. ENCODER LAYER
# POST-LN
# ============================================================

class TransformerEncoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x):

        # Self-attention -> residual -> LayerNorm
        x = self.norm1(
            x + self.dropout(
                self.self_attn(x, x)
            )
        )

        # FFN -> residual -> LayerNorm
        x = self.norm2(
            x + self.ffn(x)
        )

        return x


# ============================================================
# 6. ENCODER
# ============================================================

class TransformerEncoder(nn.Module):
    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerEncoderLayer(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

    def forward(self, x):

        for layer in self.layers:
            x = layer(x)

        return x


# ============================================================
# 7. DECODER LAYER
# THREE POST-LN SUBLAYERS
# ============================================================

class TransformerDecoderLayer(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()

        self.self_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.cross_attn = MultiHeadAttention(
            d_model=d_model,
            n_heads=n_heads,
            dropout=dropout
        )

        self.ffn = FeedForward(
            d_model=d_model,
            d_ff=d_ff,
            dropout=dropout
        )

        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.norm3 = nn.LayerNorm(d_model)

        self.dropout = nn.Dropout(dropout)

    def forward(self, x, memory, causal_mask):

        # 1. Causal self-attention
        x = self.norm1(
            x + self.dropout(
                self.self_attn(
                    x,
                    x,
                    mask=causal_mask
                )
            )
        )

        # 2. Cross-attention
        # Q = decoder, K/V = encoder memory
        x = self.norm2(
            x + self.dropout(
                self.cross_attn(
                    x,
                    memory
                )
            )
        )

        # 3. FFN
        x = self.norm3(
            x + self.ffn(x)
        )

        return x


# ============================================================
# 8. DECODER
# ============================================================

class TransformerDecoder(nn.Module):
    def __init__(
        self,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        max_len=1024,
        dropout=0.1
    ):
        super().__init__()

        self.layers = nn.ModuleList([
            TransformerDecoderLayer(
                d_model=d_model,
                n_heads=n_heads,
                d_ff=d_ff,
                dropout=dropout
            )
            for _ in range(n_layers)
        ])

        self.register_buffer(
            "causal_mask",
            make_causal_mask(max_len),
            persistent=False
        )

    def forward(self, x, memory):

        T = x.size(1)

        mask = self.causal_mask[:T, :T]

        for layer in self.layers:
            x = layer(
                x,
                memory,
                mask
            )

        return x


# ============================================================
# 9. COMPLETE ENCODER-DECODER MODEL
# ============================================================

class TransformerModel(nn.Module):
    def __init__(
        self,
        vocab_size,
        max_src_len,
        max_tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    ):
        super().__init__()

        self.vocab_size = vocab_size
        self.d_model = d_model
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

        # Shared token embedding
        self.token_embedding = nn.Embedding(
            vocab_size,
            d_model
        )

        # Fixed sinusoidal positional encodings
        self.register_buffer(
            "src_pos_encoding",
            make_sinusoidal_table(
                max_src_len,
                d_model
            ),
            persistent=False
        )

        self.register_buffer(
            "tgt_pos_encoding",
            make_sinusoidal_table(
                max_tgt_len,
                d_model
            ),
            persistent=False
        )

        self.dropout = nn.Dropout(dropout)

        self.encoder = TransformerEncoder(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            dropout=dropout
        )

        self.decoder = TransformerDecoder(
            d_model=d_model,
            n_heads=n_heads,
            d_ff=d_ff,
            n_layers=n_layers,
            max_len=max_tgt_len,
            dropout=dropout
        )

        # No bias: exact tied vocabulary projection
        self.output_projection = nn.Linear(
            d_model,
            vocab_size,
            bias=False
        )

        # Tie output projection to token embedding
        self.output_projection.weight = self.token_embedding.weight

        self._init_weights()

    def _init_weights(self):

        nn.init.normal_(
            self.token_embedding.weight,
            mean=0.0,
            std=0.02
        )

        for module in self.modules():

            if isinstance(module, nn.Linear):

                if module.weight is not self.token_embedding.weight:
                    nn.init.xavier_uniform_(module.weight)

                if module.bias is not None:
                    nn.init.zeros_(module.bias)

            elif isinstance(module, nn.LayerNorm):

                nn.init.ones_(module.weight)
                nn.init.zeros_(module.bias)

    def encode(self, src):

        B, T_src = src.shape

        assert T_src <= self.max_src_len

        # Original Transformer-style sqrt(d) scaling
        x = self.token_embedding(src)
        x = x * math.sqrt(self.d_model)

        # Add fixed positional encoding
        x = x + self.src_pos_encoding[:T_src].unsqueeze(0)

        x = self.dropout(x)

        return self.encoder(x)

    def decode(self, tgt_in, memory):

        B, T_tgt = tgt_in.shape

        assert T_tgt <= self.max_tgt_len

        # Token embedding + sqrt(d) scaling
        x = self.token_embedding(tgt_in)
        x = x * math.sqrt(self.d_model)

        # Add fixed positional encoding
        x = x + self.tgt_pos_encoding[:T_tgt].unsqueeze(0)

        x = self.dropout(x)

        x = self.decoder(
            x,
            memory
        )

        # Raw logits; no softmax
        return self.output_projection(x)

    def forward(self, src, tgt_in):

        memory = self.encode(src)

        return self.decode(
            tgt_in,
            memory
        )


# ============================================================
# 10. BUILD_MODEL
# ============================================================

def build_model(vocab_size, src_len, tgt_len):

    assert src_len + tgt_len == 1024

    return TransformerModel(
        vocab_size=vocab_size,
        max_src_len=src_len,
        max_tgt_len=tgt_len,
        d_model=768,
        n_heads=8,
        d_ff=3072,
        n_layers=6,
        dropout=0.1
    )

# ============================================================
# 1. PATHS / CONFIG
# ============================================================
TOKD = "/kaggle/input/datasets/achiver7/phase1-tokens-v40k-clean1/tokens"
RUN = "/kaggle/working/run_arch2_encdec"          # new name per architecture
V, TIME_LIMIT_H, dev = 40000, 8.25, "cuda"

CFG = dict(
    BLOCK=1024,            # total tokens per window = SRC_LEN + TGT_LEN (same windows as Architecture 1)
    SRC_LEN=768,          # YOU: tokens given to the encoder
    TGT_LEN=256,          # YOU: tokens the decoder predicts
    MICRO_BATCH=1, GRAD_ACCUM=16,
    LR=3e-4, MIN_LR_FRAC=0.1, WD=0.1, WARMUP=500,
    STEPS=10000, CLIP=1.0, EVAL_EVERY=500, SEED=42,
    MIX=dict(acts=0.273, sc=0.469, tri=0.258),
)
missing = [k for k in ("SRC_LEN", "TGT_LEN") if CFG[k] is None]
assert not missing, f"set these in CFG first: {missing}"
BLOCK, S, TT = CFG["BLOCK"], CFG["SRC_LEN"], CFG["TGT_LEN"]
MB, GA = CFG["MICRO_BATCH"], CFG["GRAD_ACCUM"]
assert S + TT == BLOCK, "SRC_LEN + TGT_LEN must equal BLOCK so validation windows match Architecture 1"
assert abs(sum(CFG["MIX"].values()) - 1.0) < 1e-6

assert torch.cuda.is_available(), "CUDA GPU required"
assert os.path.exists(f"{TOKD}/tokenizer.json"), TOKD
tok = Tokenizer.from_file(f"{TOKD}/tokenizer.json")
BOS = tok.token_to_id("<bos>")
assert BOS is not None, "tokenizer has no <bos>"

np.random.seed(CFG["SEED"]); torch.manual_seed(CFG["SEED"]); torch.cuda.manual_seed_all(CFG["SEED"])
os.makedirs(RUN, exist_ok=True)

# ============================================================
# 2. DATA
# ============================================================
def mm(name):
    p = f"{TOKD}/{name}.bin"
    assert os.path.exists(p), p
    return np.memmap(p, dtype=np.uint16, mode="r")

data = {"acts": mm("acts_train"), "sc": mm("sc_train"), "tri": mm("tribunals_train_f")}
valbins = {"acts_val": mm("acts_val"), "sc_val": mm("sc_val"), "tribunals_val_f": mm("tribunals_val_f")}
keys, probs = list(CFG["MIX"]), list(CFG["MIX"].values())


def split_window(w):
    """w: LongTensor [B, >=BLOCK]. Returns src, tgt_in, tgt_out."""
    w = w[:, :BLOCK]
    src, tgt_out = w[:, :S], w[:, S:]
    bos = torch.full((w.size(0), 1), BOS, dtype=w.dtype, device=w.device)
    tgt_in = torch.cat([bos, tgt_out[:, :-1]], dim=1)
    return src, tgt_in, tgt_out


def get_batch():
    ws = []
    for _ in range(MB):
        d = data[keys[np.random.choice(len(keys), p=probs)]]
        i = np.random.randint(0, len(d) - BLOCK - 1)
        ws.append(d[i:i + BLOCK].astype(np.int64))
    w = torch.from_numpy(np.stack(ws)).to(dev)
    return split_window(w)


# SAME fixed windows as Architecture 1 (RandomState 123, BLOCK+1 tokens each)
rs = np.random.RandomState(123)
val_windows = {}
for name, d in valbins.items():
    starts = rs.randint(0, len(d) - BLOCK - 1, 200)
    val_windows[name] = np.stack([d[i:i + BLOCK + 1].astype(np.int64) for i in starts])

# ============================================================
# 3. MODEL + PREFLIGHT
# ============================================================
model = build_model(V, S, TT).to(dev)
n_params = sum(p.numel() for p in model.parameters())
print(f"PARAMETERS: {n_params:,} ({n_params/1e6:.2f}M)")
assert 50e6 <= n_params <= 300e6, f"Invalid parameter count: {n_params/1e6:.2f}M"


@torch.no_grad()
def preflight():
    model.eval()
    src = torch.randint(0, V, (2, S), device=dev)
    tin = torch.randint(0, V, (2, TT), device=dev)
    out = model(src, tin).float()
    assert out.shape == (2, TT, V), f"bad output shape {tuple(out.shape)}"
    print(f"[preflight] shape OK: {tuple(out.shape)}")

    t = TT // 2                                   # decoder causality
    tin2 = tin.clone(); tin2[:, t:] = torch.randint(0, V, tin2[:, t:].shape, device=dev)
    leak = (out[:, :t] - model(src, tin2).float()[:, :t]).abs().max().item()
    print(f"[preflight] decoder: max logit change before position {t}: {leak:.2e}")
    assert leak < 1e-3, "FUTURE DECODER TOKENS LEAK: decoder self-attention mask is wrong"

    src2 = torch.randint(0, V, (2, S), device=dev)   # cross-attention is wired up
    dep = (out - model(src2, tin).float()).abs().max().item()
    print(f"[preflight] max logit change when ONLY the encoder input changes: {dep:.2e}")
    assert dep > 0, "decoder output ignores the encoder input: cross-attention is not connected"

    s, a, b = get_batch()
    with torch.autocast("cuda", dtype=torch.float16):
        lg = model(s, a)
    l0 = F.cross_entropy(lg.float().reshape(-1, V), b.reshape(-1)).item()
    print(f"[preflight] initial loss {l0:.2f} (uniform guess = {math.log(V):.2f})")
    model.train()


preflight()

# ============================================================
# 4. OPTIMIZER, SCHEDULE, STEP, VALIDATION, CHECKPOINT
# ============================================================
opt = torch.optim.AdamW(model.parameters(), lr=CFG["LR"], weight_decay=CFG["WD"])
scaler = torch.amp.GradScaler("cuda")


def lr_at(step):
    if step < CFG["WARMUP"]:
        return CFG["LR"] * (step + 1) / CFG["WARMUP"]
    p = min(max((step - CFG["WARMUP"]) / max(1, CFG["STEPS"] - CFG["WARMUP"]), 0.0), 1.0)
    return CFG["LR"] * (CFG["MIN_LR_FRAC"] + (1 - CFG["MIN_LR_FRAC"]) * 0.5 * (1 + math.cos(math.pi * p)))


def train_step(lr):
    for g in opt.param_groups:
        g["lr"] = lr
    total = 0.0
    for _ in range(GA):
        src, tin, tout = get_batch()
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(src, tin)
        loss = F.cross_entropy(lg.float().reshape(-1, V), tout.reshape(-1)) / GA
        scaler.scale(loss).backward()
        total += loss.item()
        del src, tin, tout, lg, loss
    scaler.unscale_(opt)
    gn = nn.utils.clip_grad_norm_(model.parameters(), CFG["CLIP"])
    scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True)
    return total, float(gn)


@torch.no_grad()
def val_loss(bs=1):
    """Mean loss per target token (the last TGT_LEN tokens of each fixed window)."""
    model.eval(); res = {}
    for name, w in val_windows.items():
        tot = cnt = 0
        for i in range(0, len(w), bs):
            b = torch.from_numpy(w[i:i + bs]).to(dev)
            src, tin, tout = split_window(b)
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model(src, tin)
            tot += F.cross_entropy(lg.float().reshape(-1, V), tout.reshape(-1), reduction="sum").item()
            cnt += tout.numel()
        res[name] = tot / cnt
    model.train()
    return res


def save_ckpt(path, step, best):
    torch.save({"model": model.state_dict(), "opt": opt.state_dict(), "scaler": scaler.state_dict(),
                "step": step, "best": best, "cfg": CFG, "n_params": n_params}, path)


def run_training():
    step0, best = 0, float("inf")
    latest = f"{RUN}/latest.pt"
    if os.path.exists(latest):
        ck = torch.load(latest, map_location=dev)
        model.load_state_dict(ck["model"]); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        step0, best = ck["step"], ck["best"]
        if ck.get("cfg") != CFG:
            print("WARNING: CFG differs from the checkpoint's CFG")
        np.random.seed(CFG["SEED"] + step0)
        print("RESUMED at step", step0, "| best", best)
    else:
        print("STARTING FROM FRESH WEIGHTS")
    model.train()
    start = last_t = time.time(); last_s = step0; hist = []
    for step in range(step0, CFG["STEPS"]):
        lr = lr_at(step)
        loss, gn = train_step(lr); hist.append(loss)
        timeout = time.time() - start >= TIME_LIMIT_H * 3600
        if (step + 1) % CFG["EVAL_EVERY"] == 0 or step + 1 == CFG["STEPS"] or timeout:
            val = val_loss(); vm = float(np.mean(list(val.values())))
            now = time.time()
            tps = (step + 1 - last_s) * MB * GA * BLOCK / max(now - last_t, 1e-9)
            seen = (step + 1) * MB * GA * BLOCK
            print(f"\nSTEP {step+1:,}/{CFG['STEPS']:,} | {seen/1e6:,.1f}M tok consumed | train {np.mean(hist[-50:]):.4f} | "
                  + " ".join(f"{k}={v:.4f}" for k, v in val.items())
                  + f" | mean {vm:.4f} | {tps:,.0f} tok/s | {torch.cuda.max_memory_allocated()/1e9:.2f} GB")
            rec = {"step": step + 1, "tokens_consumed": int(seen),
                   "target_tokens": int((step + 1) * MB * GA * TT),
                   "train_loss": float(np.mean(hist[-50:])), "val_mean": vm, "lr": float(lr),
                   "grad_norm": gn, "tok_per_sec": float(tps), **val}
            with open(f"{RUN}/log.jsonl", "a") as f:
                f.write(json.dumps(rec) + "\n")
            if vm < best:
                best = vm; save_ckpt(f"{RUN}/best.pt", step + 1, best)
            save_ckpt(latest, step + 1, best)
            last_t, last_s = now, step + 1
        if timeout:
            print("TIME GUARD: checkpoint saved. Resume with the same RUN path.")
            return "PAUSED"
    return "DONE"


# ============================================================
# 5. COMPARABLE EVALUATION OF ARCHITECTURE 1 (optional helper)
#    Scores a decoder-only model on exactly the same target tokens
#    (the last TGT_LEN tokens of each fixed window), so the two
#    architectures' validation losses are directly comparable.
# ============================================================
@torch.no_grad()
def decoder_only_loss_on_targets(model1, bs=1):
    model1.eval(); res = {}
    for name, w in val_windows.items():
        tot = cnt = 0
        for i in range(0, len(w), bs):
            b = torch.from_numpy(w[i:i + bs]).to(dev)
            with torch.autocast("cuda", dtype=torch.float16):
                lg = model1(b[:, :-1])
            y = b[:, 1:]
            tot += F.cross_entropy(lg[:, S - 1:BLOCK - 1].float().reshape(-1, V),
                                   y[:, S - 1:BLOCK - 1].reshape(-1), reduction="sum").item()
            cnt += y[:, S - 1:BLOCK - 1].numel()
        res[name] = tot / cnt
    return res


# ============================================================
# 6. GENERATION HELPER (defined here; used after training)
#    Encoder reads the last SRC_LEN tokens of the prompt; the decoder
#    generates from <bos>. Needs every prompt to have >= SRC_LEN tokens
#    (test prompts: mean 609 tokens, max 918).
# ============================================================
def sample_next(lg, GEN, gen):
    if GEN["TEMP"] == 0:
        return lg.argmax(-1, keepdim=True)
    lg = lg / GEN["TEMP"]
    if GEN["TOP_K"]:
        v, _ = torch.topk(lg, min(GEN["TOP_K"], lg.size(-1)))
        lg[lg < v[:, [-1]]] = -float("inf")
    p = F.softmax(lg, -1)
    if GEN["TOP_P"] < 1.0:
        sp, si = torch.sort(p, descending=True)
        cum = sp.cumsum(-1)
        sp[(cum - sp) > GEN["TOP_P"]] = 0
        p = torch.zeros_like(p).scatter_(1, si, sp)
        p = p / p.sum(-1, keepdim=True)
    return torch.multinomial(p, 1, generator=gen)


@torch.no_grad()
def encdec_continue(model, prompt, GEN, gen):
    """Returns the generated words (list). GEN: dict(TEMP, TOP_K, TOP_P) chosen by you."""
    model.eval()
    ids = tok.encode(prompt).ids
    if len(ids) < S:
        raise ValueError(f"prompt has {len(ids)} tokens < SRC_LEN={S}")
    src = torch.tensor([ids[-S:]], device=dev)
    tgt = torch.tensor([[BOS]], device=dev)
    new = []
    while len(new) < TT:
        with torch.autocast("cuda", dtype=torch.float16):
            lg = model(src, F.pad(tgt, (0, TT - tgt.size(1)), value=BOS))[:, tgt.size(1) - 1, :].float()
        nxt = sample_next(lg, GEN, gen)
        tgt = torch.cat([tgt, nxt], 1)
        new.append(nxt.item())
        if len(new) % 8 == 0 and len(tok.decode(new).split()) >= 128:
            break
    return tok.decode(new).split()


# ============================================================
# 7. RUN
# ============================================================
if __name__ == "__main__":
    print("STARTING TRAINING...\n")
    status = run_training()
    print("\nFINAL STATUS:", status, "| RUN:", RUN)
    for f in ["best.pt", "latest.pt", "log.jsonl"]:
        print(f"{f:10s}", "OK" if os.path.exists(os.path.join(RUN, f)) else "MISSING")

PARAMETERS: 129,957,888 (129.96M)
[preflight] shape OK: (2, 256, 40000)
[preflight] decoder: max logit change before position 128: 0.00e+00
[preflight] max logit change when ONLY the encoder input changes: 2.22e-01
[preflight] initial loss 10.73 (uniform guess = 10.60)
STARTING TRAINING...

STARTING FROM FRESH WEIGHTS

STEP 500/10,000 | 8.2M tok consumed | train 6.9442 | acts_val=6.8081 sc_val=6.9387 tribunals_val_f=7.0551 | mean 6.9340 | 13,891 tok/s | 3.54 GB

STEP 1,000/10,000 | 16.4M tok consumed | train 7.0221 | acts_val=6.8901 sc_val=6.9166 tribunals_val_f=7.5029 | mean 7.1032 | 14,075 tok/s | 3.54 GB

STEP 1,500/10,000 | 24.6M tok consumed | train 7.0504 | acts_val=6.8798 sc_val=6.9128 tribunals_val_f=7.4101 | mean 7.0675 | 14,253 tok/s | 3.54 GB

STEP 2,000/10,000 | 32.8M tok consumed | train 7.0382 | acts_val=6.8715 sc_val=6.9098 tribunals_val_f=7.3850 | mean 7.0554 | 14,249 tok/s | 3.54 GB

STEP 2,500/10,000 | 41.0M tok consumed | train 7.0309 | acts_val=6.8901 sc_val=6.9168 

In [10]:
abf

NameError: name 'abf' is not defined

In [ ]:
zac